# Tutorial 21: HAD Pre-test Workflow - Did the Brand Campaign Satisfy the Identifying Assumptions?

[Tutorial 20](20_had_brand_campaign.ipynb) fit `HeterogeneousAdoptionDiD` (HAD) on a regional brand-campaign panel and reported a per-dollar lift, with a brief visual placebo check at the end. We deliberately deferred the **formal pre-test workflow** to this tutorial, with a forward pointer in T20's "Extensions" section.

This tutorial picks up where T20 left off. We re-run the brand campaign on a panel close in shape to T20's, then walk through HAD's composite pre-test workflow `did_had_pretest_workflow` to formally validate the identifying assumptions (paper Section 4.2 of de Chaisemartin, Ciccia, D'Haultfoeuille, & Knau (2026)). We start with the two-period (`aggregate="overall"`) workflow, observe that it leaves the parallel pre-trends step open, and then **upgrade** to the multi-period (`aggregate="event_study"`) workflow that closes all three paper steps jointly. A side panel compares the two `null=` modes of the Yatchew-HR linearity test, including the recently-shipped `null="mean_independence"` mode (R-parity with `YatchewTest::yatchew_test(order=0)`).


## 1. The Pre-test Battery

de Chaisemartin et al. (2026) Section 4.2 lays out a four-step workflow for HAD identification:

1. **Step 1 - QUG support-infimum test (paper Theorem 4):** is the support of the dose distribution consistent with `d_lower = 0` (Design 1, `continuous_at_zero`, target = `WAS`)? Or is the support strictly above zero (Design 1', `continuous_near_d_lower`, target = `WAS_d_lower`)? The two designs identify different estimands; getting this right matters.
2. **Step 2 - Parallel pre-trends (paper Assumption 7):** does the differenced outcome behave the same way across dose groups in the *pre-treatment* periods? Same identifying logic as classic DiD.
3. **Step 3 - Linearity / homogeneity (paper Assumption 8):** is `E[dY | D]` linear in `D`, so that the WAS reading reflects the average per-dose marginal effect rather than masking heterogeneity bias?
4. **Step 4 - Boundary continuity (paper Assumptions 5, 6):** local-linearity of the dose-response near the boundary `d_lower`. **Non-testable**; argued from domain knowledge.

The library bundles the testable steps into one entry point: `did_had_pretest_workflow`. It dispatches to a two-period implementation (steps 1 + 3 only - step 2 needs at least two pre-periods) or a multi-period implementation (steps 1 + 2 + 3 jointly). The Yatchew-HR test from Step 3 is also exposed standalone with two null modes; we exercise both in the side panel.


## 2. The Panel

We use a panel close in shape to T20's brand campaign (60 DMAs over 8 weeks, regional add-on spend on top of a national TV blast at week 5, true per-$1K lift = 100 weekly visits). The one difference: regional spend in this tutorial spans roughly **$10 to $50K** (Uniform[\$0.01K, \$50K]) instead of T20's Uniform[\$5K, \$50K]. Some markets barely participated in the regional add-on - they put in essentially nothing. This shifts HAD's design path from T20's `continuous_near_d_lower` (Design 1', target = `WAS_d_lower`) to `continuous_at_zero` (Design 1, target = `WAS`) - the QUG test in Step 1 confirms that.


In [1]:
import numpy as np
import pandas as pd

from diff_diff import generate_continuous_did_data

MAIN_SEED = 87
N_UNITS = 60
N_PERIODS = 8
COHORT_PERIOD = 5
TRUE_SLOPE = 100.0
BASELINE_VISITS = 5000.0
DOSE_LOW = 0.01
DOSE_HIGH = 50.0

raw = generate_continuous_did_data(
    n_units=N_UNITS,
    n_periods=N_PERIODS,
    cohort_periods=[COHORT_PERIOD],
    never_treated_frac=0.0,
    dose_distribution="uniform",
    dose_params={"low": DOSE_LOW, "high": DOSE_HIGH},
    att_function="linear",
    att_intercept=0.0,
    att_slope=TRUE_SLOPE,
    unit_fe_sd=8.0,
    time_trend=0.5,
    noise_sd=2.0,
    seed=MAIN_SEED,
)
panel = raw.copy()
panel.loc[panel["period"] < panel["first_treat"], "dose"] = 0.0
panel = panel.rename(
    columns={
        "unit": "dma_id",
        "period": "week",
        "outcome": "weekly_visits",
        "dose": "regional_spend_k",
    }
)
panel["weekly_visits"] = panel["weekly_visits"] + BASELINE_VISITS

post = panel[panel["week"] >= COHORT_PERIOD]
print(f"Panel: {panel['dma_id'].nunique()} DMAs x {panel['week'].nunique()} weeks")
print(
    f"Regional spend (post-launch): "
    f"${post['regional_spend_k'].min():.2f}K - "
    f"${post['regional_spend_k'].max():.2f}K"
)
print(f"True per-$1K lift (locked at seed): {TRUE_SLOPE} weekly visits")


Panel: 60 DMAs x 8 weeks
Regional spend (post-launch): $0.18K - $49.00K
True per-$1K lift (locked at seed): 100.0 weekly visits


## 3. Step 1: The Overall Workflow (Two-Period Path)

T20's headline used a two-period collapse of the panel - average pre-launch outcome per DMA against average post-launch outcome per DMA. That's also the natural input shape for HAD's two-period (`aggregate="overall"`) pre-test workflow, which runs **paper Step 1 (QUG) + paper Step 3 (linearity, via Stute and Yatchew-HR)**. Step 2 (parallel pre-trends) is not implemented on this path - a single pre-period structurally can't support a pre-trends test - and the workflow's verdict says so explicitly.

We collapse to two periods (pre = avg over weeks 1-4, post = avg over weeks 5-8), then call the workflow.


In [2]:
from diff_diff import did_had_pretest_workflow

p = panel.copy()
p["period"] = (p["week"] >= COHORT_PERIOD).astype(int) + 1  # 1=pre, 2=post
two_period = p.groupby(["dma_id", "period"], as_index=False).agg(
    weekly_visits=("weekly_visits", "mean"),
    regional_spend_k=("regional_spend_k", "mean"),
)
# Workflow invariant: pre-period dose = 0 for every unit.
two_period.loc[two_period["period"] == 1, "regional_spend_k"] = 0.0
# first_treat in the collapsed coordinates: 2 (the post-period) for every DMA.
two_period["first_treat"] = 2

overall_report = did_had_pretest_workflow(
    data=two_period,
    outcome_col="weekly_visits",
    dose_col="regional_spend_k",
    time_col="period",
    unit_col="dma_id",
    first_treat_col="first_treat",
    alpha=0.05,
    n_bootstrap=999,
    seed=21,
    aggregate="overall",
)

print(overall_report.verdict)
print(f"\nall_pass = {overall_report.all_pass}")
print(f"aggregate = {overall_report.aggregate!r}")
print(f"pretrends_joint populated? {overall_report.pretrends_joint is not None}")
print(f"homogeneity_joint populated? {overall_report.homogeneity_joint is not None}")


QUG and linearity diagnostics fail-to-reject; Assumption 7 pre-trends test NOT run (paper step 2 deferred to Phase 3 follow-up)

all_pass = True
aggregate = 'overall'
pretrends_joint populated? False
homogeneity_joint populated? False


**Reading the overall verdict.** Three things to note.

- **Step 1 (QUG) fails to reject:** `D_(1)` (the smallest treated dose, ~\$180 here) is small relative to the gap `D_(2) - D_(1)`, so the test statistic `T = D_(1) / (D_(2) - D_(1))` lands well below its critical value (1/alpha - 1 = 19 at alpha = 0.05). The data are consistent with `d_lower = 0` (Design 1, `continuous_at_zero`, target = `WAS`).
- **Step 3 (linearity) fails to reject** on both Stute (CvM) and Yatchew-HR. The differenced outcome `dY` looks linear in `D`, so the WAS reading reflects the average per-dose marginal effect rather than masking heterogeneity bias.
- **Step 2 (Assumption 7 pre-trends) is structurally absent.** The verdict says so verbatim: `"Assumption 7 pre-trends test NOT run (paper step 2 deferred to Phase 3 follow-up)"`. With a single pre-period (the avg over weeks 1-4), there is nothing to compare against - we need at least two pre-periods to run a parallel-trends test on the dose dimension. The structural fields back this up: `pretrends_joint` and `homogeneity_joint` on the report are both `None` (the joint-Stute output containers don't get populated on the two-period path).

Let's look at each individual test result.


In [3]:
overall_report.qug.print_summary()
print()
overall_report.stute.print_summary()
print()
overall_report.yatchew.print_summary()


                QUG null test (H_0: d_lower = 0)                
Statistic T:                                 3.8562
p-value:                                     0.2059
Critical value (1/alpha-1):                 19.0000
Reject H_0:                                   False
alpha:                                       0.0500
Observations:                                    60
Excluded (d == 0):                                0
D_(1):                                       0.1806
D_(2):                                       0.2274

         Stute CvM linearity test (H_0: linear E[dY|D])         
CvM statistic:                               0.0735
Bootstrap p-value:                           0.6860
Reject H_0:                                   False
alpha:                                       0.0500
Bootstrap replications:                         999
Observations:                                    60
Seed:                                            21

        Yatchew-HR linearity test (H

A note on the Yatchew row. The `T_hr` statistic is **very large and negative** (~-35,000). That looks alarming but is correct here: under perfectly linear dose-response with very heterogeneous doses (Uniform[\$0.01K, \$50K]) and 60 sorted-by-dose units, the differencing variance `sigma2_diff` (which captures the squared gap between adjacent-by-dose units' `dy` values) is much larger than the OLS residual variance `sigma2_lin`. The formula `T_hr = sqrt(G) * (sigma2_lin - sigma2_diff) / sigma2_W` then goes massively negative, p-value rounds to 1.0, and we comfortably fail to reject linearity. (For a different way to look at this same test, see the Yatchew side panel later in the notebook.)


## 4. Step 2: Upgrade to the Event-Study Workflow

The two-period workflow gave us evidence on Steps 1 and 3 but no formal evidence on Step 2 (parallel pre-trends). Our panel actually has 8 weeks - that's enough pre-periods to close Step 2 jointly with Stute's joint variant (paper Section 4.2 step 2 + Hlavka-Huskova 2020 / Delgado-Manteiga 2001 dependence-preserving Mammen multiplier bootstrap).

We pass the full multi-period panel to `did_had_pretest_workflow(aggregate="event_study", ...)`. The dispatch covers all three paper steps in one call:

- **Step 1**: QUG re-runs on the dose distribution at the treatment period `F` (deterministic; same numbers as the overall path).
- **Step 2**: `joint_pretrends_test` - mean-independence joint Stute over the pre-period horizons (`E[Y_t - Y_base | D] = mu_t` for each t < F).
- **Step 3**: `joint_homogeneity_test` - linearity joint Stute over the post-period horizons (`E[Y_t - Y_base | D_t] = beta_{0,t} + beta_{fe,t} * D` for each t >= F).

Step 3's "Yatchew-HR" arm has no joint variant in the paper (the differencing-based variance estimator doesn't have a derived multi-horizon extension), so the event-study path runs only joint Stute for linearity. Practitioners who want Yatchew-HR robustness on multi-period data can call the standalone `yatchew_hr_test` on each (base, post) pair manually.


In [4]:
es_report = did_had_pretest_workflow(
    data=panel,
    outcome_col="weekly_visits",
    dose_col="regional_spend_k",
    time_col="week",
    unit_col="dma_id",
    first_treat_col="first_treat",
    alpha=0.05,
    n_bootstrap=999,
    seed=21,
    aggregate="event_study",
)

print(es_report.verdict)
print(f"\nall_pass = {es_report.all_pass}")
print(f"aggregate = {es_report.aggregate!r}")
print(f"pretrends_joint populated? {es_report.pretrends_joint is not None}")
print(f"homogeneity_joint populated? {es_report.homogeneity_joint is not None}")


QUG, joint pre-trends, and joint linearity diagnostics fail-to-reject (TWFE admissible under Section 4 assumptions)

all_pass = True
aggregate = 'event_study'
pretrends_joint populated? True
homogeneity_joint populated? True


**Reading the event-study verdict.** Now the verdict reads `"QUG, joint pre-trends, and joint linearity diagnostics fail-to-reject (TWFE admissible under Section 4 assumptions)"`. The `"deferred"` caveat from the overall path is gone - all three paper steps closed jointly. The structural fields confirm: `pretrends_joint` and `homogeneity_joint` are both populated.

The joint pre-trends test runs over `n_horizons = 3` (pre-periods 1, 2, 3, with week 4 reserved as the base period). The joint homogeneity test runs over `n_horizons = 4` (post-periods 5, 6, 7, 8). Let's inspect the per-horizon detail.


In [5]:
es_report.qug.print_summary()
print()
es_report.pretrends_joint.print_summary()
print()
es_report.homogeneity_joint.print_summary()


                QUG null test (H_0: d_lower = 0)                
Statistic T:                                 3.8562
p-value:                                     0.2059
Critical value (1/alpha-1):                 19.0000
Reject H_0:                                   False
alpha:                                       0.0500
Observations:                                    60
Excluded (d == 0):                                0
D_(1):                                       0.1806
D_(2):                                       0.2274

     Joint Stute CvM test (mean-independence (pre-trends))      
Joint CvM statistic:                         7.1627
Bootstrap p-value:                           0.0720
Reject H_0:                                   False
alpha:                                       0.0500
Bootstrap replications:                         999
Horizons:                                         3
Observations:                                    60
Seed:                                

The pre-trends p-value (~0.07) sits close to the conventional alpha = 0.05 threshold - the test is not vacuous, it is informative. It is consistent with parallel pre-trends but not by a wide margin. In a real analysis this would warrant a closer look at the per-horizon CvM contributions (visible in `per_horizon_stats`) and possibly a Pierce-Schott-style linear-trend detrending via `trends_lin=True` (an extension we do not demonstrate here; see `did_had_pretest_workflow`'s docstring).

The joint homogeneity p-value (~0.76) is a strong fail-to-reject. Linearity holds across all four post-launch horizons.

Together with QUG (design verdict) and joint linearity (Step 3), this closes the testable portion of the paper's identification framework. Step 4 (boundary continuity, Assumptions 5 / 6) remains non-testable; we still defend it from domain knowledge as in T20.


## 5. Side Panel: Yatchew-HR Null Modes

The Yatchew-HR test exposes two `null=` modes (the second was added in 2026-04 for parity with the R `YatchewTest` package).

- `null="linearity"` (default; paper Theorem 7): tests `H0: E[dY | D]` is linear in `D`. Residuals come from OLS `dy ~ 1 + d`. This is what `did_had_pretest_workflow` calls under the hood.
- `null="mean_independence"` (PR #400, 2026-04, Phase 4 R-parity): tests the stricter `H0: E[dY | D] = E[dY]`, i.e. `dY` is mean-independent of `D`. Residuals come from intercept-only OLS `dy ~ 1`. Mirrors R `YatchewTest::yatchew_test(order=0)`.

The mean-independence mode is typically used on **placebo (pre-treatment) data** to test parallel pre-trends as a non-parametric mean-independence assertion. Below we construct an illustrative input - the within-pre-period first-difference `dy = Y[week=4] - Y[week=3]` paired with each DMA's actual post-period dose - and run both modes side by side. Both should fail to reject on this clean linear DGP; the contrast is in the residual structure.


In [6]:
from diff_diff import yatchew_hr_test

panel_sorted = panel.sort_values(["dma_id", "week"]).reset_index(drop=True)
pre = panel_sorted[panel_sorted["week"].isin([3, 4])]
pre_pivot = pre.pivot(index="dma_id", columns="week", values="weekly_visits")
dy = (pre_pivot[4] - pre_pivot[3]).to_numpy(dtype=np.float64)
post_dose = (
    panel_sorted[panel_sorted["week"] == 5]
    .set_index("dma_id")
    .sort_index()["regional_spend_k"]
    .to_numpy(dtype=np.float64)
)

res_lin = yatchew_hr_test(d=post_dose, dy=dy, alpha=0.05, null="linearity")
res_mi = yatchew_hr_test(d=post_dose, dy=dy, alpha=0.05, null="mean_independence")

print(res_lin.summary())
print()
print(res_mi.summary())


        Yatchew-HR linearity test (H_0: linear E[dY|D])         
T_hr statistic:                              0.0207
p-value:                                     0.4917
Critical value (1-sided z):                  1.6449
Reject H_0:                                   False
alpha:                                       0.0500
sigma^2_lin (OLS):                           6.5340
sigma^2_diff (Yatchew):                      6.5170
sigma^2_W (HR scale):                        6.3639
Observations:                                    60

    Yatchew-HR mean-independence test (H_0: E[dY|D] = E[dY])    
T_hr statistic:                              0.5536
p-value:                                     0.2899
Critical value (1-sided z):                  1.6449
Reject H_0:                                   False
alpha:                                       0.0500
sigma^2_lin (OLS):                           7.0076
sigma^2_diff (Yatchew):                      6.5170
sigma^2_W (HR scale):                

**Reading the side-panel comparison.**

- The `linearity` mode fits `dy ~ 1 + d` and computes residual variance `sigma2_lin` from those residuals. Under a clean linear DGP the residuals are small (close to noise variance), the gap `sigma2_lin - sigma2_diff` is near zero, and `T_hr` lands close to zero with a p-value far above alpha.
- The `mean_independence` mode fits intercept-only `dy ~ 1` and computes `sigma2_lin` as the population variance of `dy`. That residual variance is **strictly larger** than under `linearity` (the linear fit absorbs the dose-response signal that intercept-only does not). The gap `sigma2_lin - sigma2_diff` is then larger and `T_hr` is larger - same asymptotic distribution, stricter null, more easily rejected when the alternative is true.

On clean linear placebo data both modes fail to reject - exactly what we want. On data where `dY` actually responds to `D` in pre-period (parallel pre-trends fail), `null="mean_independence"` is more sensitive than `null="linearity"` because linearity is a weaker null (linear pre-trends would fail to reject the linearity null but would reject the mean-independence null).

When to choose which: use `null="linearity"` to defend the joint identification assumption (paper Step 3, Assumption 8). Use `null="mean_independence"` on placebo (pre-treatment) data when you want a non-parametric mean-independence assertion. The `null="mean_independence"` mode is what R `YatchewTest::yatchew_test(order=0)` runs by default for placebo pre-trend tests.


## 6. Communicating the Validation to Leadership

Pre-test results travel awkwardly to non-technical audiences. The template below structures the validation around what each test rules out - mirroring the headline-and-evidence pattern from T20 Section 5.

> **Identifying assumptions for HAD on the brand-campaign panel are defended on all three paper steps.**
>
> - **Step 1 (QUG support-infimum, paper Theorem 4):** the test is consistent with the dose distribution starting at zero (`d_lower = 0`, p approximately 0.21). The library auto-detects the `continuous_at_zero` design and reports the WAS (Weighted Average Slope), as expected for this panel where some markets barely participated in the regional spend.
> - **Step 2 (parallel pre-trends, Assumption 7):** the joint Stute pre-trends test fails to reject (joint p approximately 0.07 across the three pre-period horizons). The pre-trend evidence is not a slam dunk - the p-value is close to alpha = 0.05 - but it is conclusive. In a high-stakes deployment we would inspect the per-horizon contributions (`per_horizon_stats`) and consider Pierce-Schott-style linear-trend detrending.
> - **Step 3 (linearity, Assumption 8):** joint Stute homogeneity fails to reject (joint p approximately 0.76 across the four post-launch horizons). The linearity assumption needed for the WAS reading to reflect the average per-dose marginal effect (rather than masking heterogeneity bias) is comfortably supported.
>
> **Non-testable from data (Step 4, paper Assumptions 5 / 6, boundary continuity):** local-linearity of the dose-response near `d_lower`. Argued from domain knowledge - is there reason to believe the marginal effect of an additional $1K of regional spend is roughly constant across the dose range? In our case yes, by DGP construction; in a real analysis we would justify this from prior knowledge of the channel's response shape.
>
> **Bottom line:** TWFE is admissible under the paper's framework on this panel. The headline per-$1K lift from the HAD fit can be carried forward to leadership without methodological caveat beyond Step 4 (which is qualitative, not data-driven).


## 7. Extensions

This tutorial covered the composite pre-test workflow on a single Design 1 panel. A few directions we did not exercise here:

- **Survey-weighted / population-weighted inference** - HAD's pre-test workflow accepts `survey_design=` (or the deprecated `survey=` / `weights=` aliases) for design-based inference. The QUG step is permanently deferred under survey weighting (extreme-value theory under complex sampling is not a settled toolkit); the linearity family runs with PSU-level Mammen multiplier bootstrap (Stute and joint variants) and weighted OLS + weighted variance components (Yatchew). A follow-up tutorial covers this path end-to-end.
- **`trends_lin=True` (Pierce-Schott Eq 17 / 18 detrending)** - mirrors R `DIDHAD::did_had(..., trends_lin=TRUE)`. Forwards into both joint pre-trends and joint homogeneity wrappers; consumes the placebo at `base_period - 1` and skips Step 2 if no earlier placebo survives the drop. Useful when you suspect linear time trends correlated with dose but want to keep the joint-Stute machinery.
- **Standalone constituent tests** - all four building blocks are exposed for direct calling: `qug_test`, `stute_test`, `yatchew_hr_test` (used in this tutorial's side panel), and the joint variants `stute_joint_pretest`, `joint_pretrends_test`, `joint_homogeneity_test`.

See the [`HeterogeneousAdoptionDiD` API reference](../api/had.html) and the [`HAD pre-tests` reference](../api/had.html#pre-tests) for the full parameter lists.

**Related tutorials.**

- [Tutorial 14: Continuous DiD](14_continuous_did.ipynb) - the Callaway-Goodman-Bacon-Sant'Anna estimator for continuous-dose settings WHERE you do have a never-treated unit AND want the per-dose ATT(d) curve, not just the average slope.
- [Tutorial 20: HAD for a National Brand Campaign](20_had_brand_campaign.ipynb) - the headline HAD fit and event-study this tutorial defends.
- [Tutorial 4: Parallel Trends](04_parallel_trends.ipynb) - parallel-trends tests for the binary-DiD setting.


## 8. Summary Checklist

- HAD's pre-test workflow `did_had_pretest_workflow` bundles paper Section 4.2 Steps 1 (QUG support infimum), 2 (joint Stute pre-trends - event-study path only), and 3 (Stute / Yatchew-HR linearity, joint variant on event-study path).
- The two-period (`aggregate="overall"`) path runs Steps 1 + 3 only - it cannot run Step 2 because a single pre-period structurally has nothing to test against. The verdict says so verbatim: "Assumption 7 pre-trends test NOT run".
- Upgrade to the multi-period (`aggregate="event_study"`) path to close all three testable steps jointly. The verdict then reads "TWFE admissible under Section 4 assumptions" when nothing rejects.
- Step 4 (paper Assumptions 5 / 6, boundary continuity) is **non-testable** from data - argue from domain knowledge.
- The Yatchew-HR test exposes two null modes: `null="linearity"` (paper Theorem 7, default; what the workflow calls under the hood) and `null="mean_independence"` (Phase 4 R-parity with R `YatchewTest::yatchew_test(order=0)`, useful on placebo pre-period data).
- Bootstrap p-values are RNG-dependent. The drift test for this notebook lives in `tests/test_t21_had_pretest_workflow_drift.py` and uses tolerance bands per backend (Rust vs pure-Python).
